# Asymmetric Cross-Modal Attention for VQA — Colab Notebook

Self-contained notebook: install deps, load VQA data from HuggingFace, define models, train and evaluate. Ready to run on Google Colab.

## 1. Install dependencies

In [1]:
!pip install -q torch torchvision transformers datasets pillow tqdm

## 2. Imports and device

In [2]:
import json
import math
import random
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 3. Load VQA dataset from HuggingFace

Uses **merve/vqav2-small** (VQA v2 style: `image`, `question`, `multiple_choice_answer`). No local files needed.

In [3]:
from datasets import load_dataset

# VQAv2-style dataset with image column (PIL); ~21k examples, we take a subset for speed
SUBSET_SIZE = 1500  # increase for full run
try:
    vqa = load_dataset("merve/vqav2-small", split="validation")
    vqa = vqa.select(range(min(SUBSET_SIZE, len(vqa))))
    print(f"Loaded {len(vqa)} examples from merve/vqav2-small")
except Exception as e:
    print("Fallback: creating minimal synthetic VQA data so the notebook runs.", e)
    vqa = []
    for i in range(min(SUBSET_SIZE, 400)):
        vqa.append({
            "image": Image.new("RGB", (224, 224), (128, 128, 128)),
            "question": "What color is the sky?" if i % 2 == 0 else "How many?",
            "multiple_choice_answer": "blue" if i % 2 == 0 else "two",
        })
    print(f"Using {len(vqa)} synthetic samples")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/403 [00:00<?, ?B/s]

data/validation-00000-of-00007.parquet:   0%|          | 0.00/484M [00:00<?, ?B/s]

data/validation-00001-of-00007.parquet:   0%|          | 0.00/486M [00:00<?, ?B/s]

data/validation-00002-of-00007.parquet:   0%|          | 0.00/483M [00:00<?, ?B/s]

...

data/validation-00004-of-00007.parquet:   0%|          | 0.00/475M [00:00<?, ?B/s]

data/validation-00005-of-00007.parquet:   0%|          | 0.00/484M [00:00<?, ?B/s]

data/validation-00006-of-00007.parquet:   0%|          | 0.00/479M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/21435 [00:00<?, ? examples/s]

Loaded 1500 examples from merve/vqav2-small


In [4]:
# Normalize to list of dicts with keys: image (PIL), question, answer
if hasattr(vqa, "__getitem__") and not isinstance(vqa, list):
    vqa_list = [vqa[i] for i in range(len(vqa))]
else:
    vqa_list = list(vqa) if hasattr(vqa, "__iter__") else vqa

# Handle both HF rows and fallback dicts
def get_image(row):
    img = row.get("image")
    if img is None:
        return Image.new("RGB", (224, 224), (128, 128, 128))
    if isinstance(img, Image.Image):
        return img.convert("RGB")
    # If it's a path-like
    try:
        return Image.open(img).convert("RGB")
    except Exception:
        return Image.new("RGB", (224, 224), (128, 128, 128))

def get_answer(row):
    a = row.get("multiple_choice_answer") or row.get("answer") or row.get("answers")
    if isinstance(a, list):
        a = a[0].get("answer", str(a[0])) if a and isinstance(a[0], dict) else str(a[0]) if a else "unknown"
    return str(a).strip().lower() if a else "unknown"

vqa_samples = []
for row in vqa_list:
    if not isinstance(row, dict):
        continue
    vqa_samples.append({
        "image": get_image(row),
        "question": str(row.get("question", "")).strip(),
        "answer": get_answer(row),
    })

if not vqa_samples:
    raise RuntimeError("No samples prepared. Expected keys: image, question, multiple_choice_answer.")
print(f"Prepared {len(vqa_samples)} samples. Example: {vqa_samples[0]['question'][:50]} -> {vqa_samples[0]['answer']}")

Prepared 1500 samples. Example: Where are the kids riding? -> carnival ride


## 4. Answer vocabulary and PyTorch Dataset

In [5]:
def build_answer_vocab(samples: List[Dict], top_k: int = 500) -> Tuple[Dict[str, int], Dict[int, str]]:
    c = Counter(s["answer"] for s in samples)
    most_common = [a for a, _ in c.most_common(top_k)]
    answer_to_idx = {a: i for i, a in enumerate(most_common)}
    idx_to_answer = {i: a for a, i in answer_to_idx.items()}
    return answer_to_idx, idx_to_answer


answer_to_idx, idx_to_answer = build_answer_vocab(vqa_samples, top_k=500)
num_answers = len(answer_to_idx)
print(f"Answer vocabulary size: {num_answers}")

Answer vocabulary size: 499


In [6]:
import torchvision.transforms as T


class VQADatasetFromSamples(Dataset):
    def __init__(
        self,
        samples: List[Dict],
        answer_to_idx: Dict[str, int],
        tokenizer_name: str = "roberta-base",
        max_length: int = 64,
        image_size: int = 224,
    ):
        self.samples = [s for s in samples if s["answer"] in answer_to_idx]
        self.answer_to_idx = answer_to_idx
        self.num_answers = len(answer_to_idx)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.max_length = max_length
        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Dict[str, torch.Tensor], int]:
        s = self.samples[idx]
        img = s["image"]
        if not isinstance(img, Image.Image):
            img = Image.open(img).convert("RGB")
        image_tensor = self.transform(img)
        enc = self.tokenizer(
            s["question"],
            padding="max_length",
            max_length=self.max_length,
            truncation=True,
            return_tensors="pt",
        )
        answer_idx = self.answer_to_idx[s["answer"]]
        return image_tensor, {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0)}, answer_idx


train_size = int(0.85 * len(vqa_samples))
train_samples = vqa_samples[:train_size]
val_samples = vqa_samples[train_size:]
train_ds = VQADatasetFromSamples(train_samples, answer_to_idx)
val_ds = VQADatasetFromSamples(val_samples, answer_to_idx)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train batches: 40, Val batches: 8


## 5. Model: Encoders

In [7]:
class ImageEncoder(nn.Module):
    def __init__(self, embed_dim: int = 512, freeze: bool = True, pretrained: bool = True):
        super().__init__()
        weights = models.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.vit_b_16(weights=weights)
        self.backbone.heads = nn.Identity()
        hidden_size = 768
        self.proj = nn.Linear(hidden_size, embed_dim) if hidden_size != embed_dim else nn.Identity()
        self.embed_dim = embed_dim
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        x = self.backbone._process_input(images)
        x = torch.cat((self.backbone.class_token.expand(x.shape[0], -1, -1), x), dim=1)
        # Encoder adds pos_embedding internally; do not add it manually
        x = self.backbone.encoder(x)
        patch_tokens = x[:, 1:, :]
        return self.proj(patch_tokens)


class TextEncoder(nn.Module):
    def __init__(self, embed_dim: int = 512, freeze: bool = True, model_name: str = "roberta-base"):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.proj = nn.Linear(hidden_size, embed_dim) if hidden_size != embed_dim else nn.Identity()
        self.embed_dim = embed_dim
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return self.proj(out.last_hidden_state)

## 6. Model: Cross-attention and fusion

In [8]:
class CrossAttentionBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, query: torch.Tensor, key_value: torch.Tensor, key_value_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, seq_q, _ = query.shape
        _, seq_kv, _ = key_value.shape
        q = self.q_proj(query).view(B, seq_q, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(key_value).view(B, seq_kv, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(key_value).view(B, seq_kv, self.num_heads, self.head_dim).transpose(1, 2)
        scale = math.sqrt(self.head_dim)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale
        if key_value_mask is not None:
            scores = scores.masked_fill(key_value_mask.unsqueeze(1).unsqueeze(2) == 0, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, seq_q, self.embed_dim)
        out = self.out_proj(out)
        return self.norm(query + self.dropout(out))


class AsymmetricCrossModalFusion(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.cross_attn_a_to_b = CrossAttentionBlock(embed_dim, num_heads, dropout)
        self.cross_attn_b_to_a = CrossAttentionBlock(embed_dim, num_heads, dropout)

    def forward(self, a: torch.Tensor, b: torch.Tensor, b_mask: Optional[torch.Tensor] = None, a_mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        a_attended = self.cross_attn_a_to_b(query=a, key_value=b, key_value_mask=b_mask)
        b_attended = self.cross_attn_b_to_a(query=b, key_value=a, key_value_mask=a_mask)
        return a_attended, b_attended

## 7. Symmetric and Asymmetric VQA models

In [9]:
class SymmetricVQAModel(nn.Module):
    def __init__(self, num_answers: int, embed_dim: int = 512, num_heads: int = 8, dropout: float = 0.3):
        super().__init__()
        self.image_encoder = ImageEncoder(embed_dim=embed_dim, freeze=True)
        self.text_encoder = TextEncoder(embed_dim=embed_dim, freeze=True)
        self.cross_attn = CrossAttentionBlock(embed_dim, num_heads=num_heads, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(embed_dim, num_answers)
        )

    def forward(self, images: torch.Tensor, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        a = self.image_encoder(images)
        b = self.text_encoder(input_ids, attention_mask)
        a_attended = self.cross_attn(query=a, key_value=b, key_value_mask=attention_mask)
        b_attended = self.cross_attn(query=b, key_value=a, key_value_mask=None)
        a_pooled = a_attended.mean(dim=1)
        b_pooled = b_attended.mean(dim=1)
        z = torch.cat([a_pooled, b_pooled], dim=-1)
        return self.classifier(z)


class AsymmetricVQAModel(nn.Module):
    def __init__(self, num_answers: int, embed_dim: int = 512, num_heads: int = 8, dropout: float = 0.3):
        super().__init__()
        self.image_encoder = ImageEncoder(embed_dim=embed_dim, freeze=True)
        self.text_encoder = TextEncoder(embed_dim=embed_dim, freeze=True)
        self.fusion = AsymmetricCrossModalFusion(embed_dim, num_heads=num_heads, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(embed_dim, num_answers)
        )

    def forward(self, images: torch.Tensor, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        a = self.image_encoder(images)
        b = self.text_encoder(input_ids, attention_mask)
        a_attended, b_attended = self.fusion(a, b, b_mask=attention_mask, a_mask=None)
        a_pooled = a_attended.mean(dim=1)
        b_pooled = b_attended.mean(dim=1)
        z = torch.cat([a_pooled, b_pooled], dim=-1)
        return self.classifier(z)

## 8. Training and evaluation

In [10]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, device: torch.device, criterion: nn.Module) -> float:
    model.train()
    total_loss, n = 0.0, 0
    for batch in loader:
        images = batch[0].to(device)
        input_ids = batch[1]["input_ids"].to(device)
        attention_mask = batch[1]["attention_mask"].to(device)
        answers = batch[2].to(device)
        optimizer.zero_grad()
        logits = model(images, input_ids, attention_mask)
        loss = criterion(logits, answers)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        n += images.size(0)
    return total_loss / n if n else 0.0


def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            images = batch[0].to(device)
            input_ids = batch[1]["input_ids"].to(device)
            attention_mask = batch[1]["attention_mask"].to(device)
            answers = batch[2]
            logits = model(images, input_ids, attention_mask)
            pred = logits.argmax(dim=1).cpu()
            correct += (pred == answers).sum().item()
            total += answers.size(0)
    return correct / total if total else 0.0

## 9. Train Asymmetric model (few epochs)

In [11]:
set_seed(42)
EMBED_DIM = 512
NUM_HEADS = 8
DROPOUT = 0.3
LR = 1e-4
EPOCHS = 3

model = AsymmetricVQAModel(num_answers=num_answers, embed_dim=EMBED_DIM, num_heads=NUM_HEADS, dropout=DROPOUT).to(device)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    loss = train_epoch(model, train_loader, optimizer, device, criterion)
    acc = evaluate(model, val_loader, device)
    print(f"Epoch {epoch + 1}/{EPOCHS}  train_loss={loss:.4f}  val_acc={acc:.2%}")

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 188MB/s]


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


AttributeError: 'VisionTransformer' object has no attribute 'encoder_pos_embedding'

## 10. (Optional) Train Symmetric baseline and compare

In [ ]:
model_sym = SymmetricVQAModel(num_answers=num_answers, embed_dim=EMBED_DIM, num_heads=NUM_HEADS, dropout=DROPOUT).to(device)
optimizer_sym = torch.optim.AdamW(filter(lambda p: p.requires_grad, model_sym.parameters()), lr=LR, weight_decay=1e-5)

for epoch in range(EPOCHS):
    loss = train_epoch(model_sym, train_loader, optimizer_sym, device, criterion)
    acc = evaluate(model_sym, val_loader, device)
    print(f"Symmetric Epoch {epoch + 1}/{EPOCHS}  train_loss={loss:.4f}  val_acc={acc:.2%}")

## 11. Summary metrics

In [ ]:
asym_acc = evaluate(model, val_loader, device)
print("Validation accuracy — Asymmetric:", f"{asym_acc:.2%}")

if "model_sym" in globals():
    sym_acc = evaluate(model_sym, val_loader, device)
    print("Validation accuracy — Symmetric:", f"{sym_acc:.2%}")
else:
    print("Symmetric model not trained (run cell 10 to compare).")